<a href="https://colab.research.google.com/github/Marcos-Med/IC/blob/main/FineTuningBERTimbau.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Realizando Fine Tuning no modelo BERTimbau
<p> Foi realizado um ajuste fino no modelo léxico BERTimbau, utilizando o corpus FakeBR. O objetivo do fine tuning é criar um modelo que preveja se uma notícia é verdadeira ou não.</p>

### Clonando o corpura que se encontra no repositório do GitHub

In [1]:
!git clone https://github.com/Marcos-Med/IC.git

Cloning into 'IC'...
remote: Enumerating objects: 21642, done.
remote: Counting objects: 100% (21642/21642), done.
remote: Compressing objects: 100% (14952/14952), done.
remote: Total 21642 (delta 6692), reused 21624 (delta 6683), pack-reused 0 (from 0)
Receiving objects: 100% (21642/21642), 21.58 MiB | 9.44 MiB/s, done.
Resolving deltas: 100% (6692/6692), done.
Updating files: 100% (21606/21606), done.


### Instalando as bibliotecas

In [2]:
!pip install transformers torch

In [3]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [4]:
# importando libs
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import pandas as pd
import os
from datasets import Dataset
from sklearn.model_selection import KFold

### Importando os dados do Corpora FakeBR

In [5]:
def load_corpus_fakeBR():
    base_path = "/content/IC/corpura/fakebr/size_normalized_texts"

    fake_news = []
    true_news = []
    for root, dirs, files in os.walk(base_path + "/fake"):
        for file in files:
            if file.endswith('.txt'):
                with open(os.path.join(root, file), 'r') as f:
                    text = f.read()
                    fake_news.append([text, 0])

    for root, dirs, files in os.walk(base_path + "/true"):
        for file in files:
            if file.endswith('.txt'):
                with open(os.path.join(root, file), 'r') as f:
                    text = f.read()
                    true_news.append([text, 1])


    all_news = fake_news + true_news
    return pd.DataFrame(all_news, columns=['text', 'label'])

In [6]:
#Carregando o corpus fakeBR
fake_br = load_corpus_fakeBR()

In [7]:
fake_br.head()

,text,label
0,Coreia do Norte volta a provocar Trump e diz q...,0
1,G20 – Trump foi irônico ao elogiar o Brasil e ...,0
2,Explosão e fogo no Vaticano deixa Europa em es...,0
3,Quem mata mais? FHC ou Bolsonaro?. A DECLARAÇ...,0
4,"Herdeira de banco doa ""tralhas"" de grife para ...",0


In [8]:
# Separar os dados textuais e a labels
text = fake_br['text']
labels = fake_br['label']

### Importando o modelo BERTimbau da NeuralMind

In [9]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = BertTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

In [10]:
#Função para tokenizar os dados para o modelo BERTimbau
def tokenize(data):
  return tokenizer(data['text'], padding='max_length', truncation=True, max_length=512)


In [11]:
#Tokeniza os dados do Corpora FakeBR
dataset = Dataset.from_pandas(fake_br)
tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

In [15]:
tokenized_dataset

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7200
})

In [16]:
# Transformar o dataset em tensor do Pytorch
tokenized_dataset.set_format("torch")

### Treinamento utilizando Validação Cruzada
**K** = 5 folds

In [17]:
# 5 folds para validação cruzada
Kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [18]:
results = []

In [19]:
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
for train_index, test_index, in Kf.split(tokenized_dataset):
    train_dataset = tokenized_dataset.select(train_index.tolist())
    test_dataset = tokenized_dataset.select(test_index.tolist())

    model = BertForSequenceClassification.from_pretrained(model_name, num_labels=2) #Classificação binária

    # Paramêtros para realizar Fine Tuning no BERTimbau
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=3,
        weight_decay=0.01,
        logging_dir="./logs",
        save_strategy="epoch"
        )
    #Objeto de treino
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset
    )

    #Treina o modelo
    trainer.train()

    #Avaliação do modelo BERTimbau
    metrics = trainer.evaluate()
    results.append(metrics)

# Converter os resultados em um DataFrame para salvar em CSV
df_results = pd.DataFrame(results)

# Salvar em um arquivo CSV
df_results.to_csv('metrics_results.csv', index=False)

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.135300,0.042021
2,0.044300,0.035694
3,0.001600,0.043791


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.147700,0.072140
2,0.034300,0.037366
3,0.003300,0.036693


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.156400,0.045658
2,0.049800,0.037381
3,0.004600,0.044640


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss
1,0.150200,0.022599
2,0.034300,0.028567


In [ ]:
model.save_pretrained("/content/IC/saveModel")
tokenizer.save_pretrained("/content/IC/saveTokenizer")

('/content/IC/saveTokenizer/tokenizer_config.json',
 '/content/IC/saveTokenizer/special_tokens_map.json',
 '/content/IC/saveTokenizer/vocab.txt',
 '/content/IC/saveTokenizer/added_tokens.json')

### Avaliação do Modelo Após Fine-Tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
saved_tokenizer = "/content/save_tokenizer"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(saved_tokenizer)

#Carrega o modelo treinado
saved_model = "/content/save_model"
model = AutoModelForSequenceClassification.from_pretrained(saved_model)

In [ ]:
def load_corpus_fakeBR():
    base_path = "/content/IC/corpura/fakebr/size_normalized_texts"

    fake_news = []
    true_news = []
    for root, dirs, files in os.walk(base_path + "/fake"):
        for file in files:
            if file.endswith('.txt'):
                with open(os.path.join(root, file), 'r') as f:
                    text = f.read()
                    fake_news.append([text, 0])

    for root, dirs, files in os.walk(base_path + "/true"):
        for file in files:
            if file.endswith('.txt'):
                with open(os.path.join(root, file), 'r') as f:
                    text = f.read()
                    true_news.append([text, 1])


    all_news = fake_news + true_news
    return pd.DataFrame(all_news, columns=['text', 'label'])

In [ ]:
!git clone https://github.com/Marcos-Med/IC.git

Cloning into 'IC'...
remote: Enumerating objects: 21636, done.
remote: Counting objects: 100% (21636/21636), done.
remote: Compressing objects: 100% (14946/14946), done.
remote: Total 21636 (delta 6689), reused 21624 (delta 6683), pack-reused 0 (from 0)
Receiving objects: 100% (21636/21636), 21.55 MiB | 12.84 MiB/s, done.
Resolving deltas: 100% (6689/6689), done.
Updating files: 100% (21605/21605), done.


In [ ]:
import os
import pandas as pd
fake_br = load_corpus_fakeBR()

In [ ]:
encodings = tokenizer(fake_br['text'].tolist(), truncation=True, padding="max_length", max_length=512)

In [ ]:
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Criação do item com as encodings
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

        # Verifique se as labels não são None antes de adicionar ao item
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # Certifique-se de que os labels são do tipo correto

        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [ ]:
labels = fake_br['label'].tolist()
labels = torch.tensor(labels, dtype=torch.long)

In [ ]:
eval_dataset = CustomDataset(encodings, labels)

In [ ]:
from transformers import Trainer, TrainingArguments

# Configuração dos argumentos de avaliação
training_args = TrainingArguments(
    output_dir="./results",
    per_device_eval_batch_size=8,  # Ajuste conforme a sua capacidade de memória
    no_cuda=True,  # Caso não tenha GPU disponível
)

# Criar o objeto Trainer
trainer = Trainer(
    model=model,  # O modelo treinado
    args=training_args,
    eval_dataset=eval_dataset  # Passando o dataset de avaliação
)

# Avaliar o modelo
metrics = trainer.evaluate()

# Imprimir ou salvar as métricas
print(metrics)

# Salvar as métricas em um arquivo JSON
import json
with open("metrics_results.json", "w") as f:
    json.dump(metrics, f, indent=4)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1583: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(
<ipython-input-30-0bff7b3375b0>:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # Certifique-se de que os labels são do tipo correto


KeyboardInterrupt: 